# fedcxr no Google Colab

Roda o experimento de aprendizado federado (ViT-S, 10 clientes, Kermany) numa GPU do Colab.

1. Em **Ambiente de execução → Alterar o tipo de ambiente**, escolha uma GPU (T4 serve).
2. Rode as células na ordem. O download do Kermany tem 1,2 GB e a preparação leva alguns minutos.
3. As execuções gravam em `runs/` e são retomáveis: se o Colab desconectar, rode a mesma célula de novo e
   as execuções com `summary.json` são puladas. Para não perder nada, monte o Google Drive (célula opcional).

A T4 não tem bf16 nativo, então as células usam `train.amp=fp16` (o código usa GradScaler nesse modo).
Isso pode mudar os números na terceira casa decimal em relação ao artigo, que usou bf16.

In [ ]:
!git clone https://github.com/fatzin/fedcxr.git
%cd fedcxr
# o Colab já traz torch com CUDA; instalamos o resto nas versões do artigo
!pip install -q timm==1.0.31 scikit-image==0.26.0 PyYAML==6.0.4
!nvidia-smi --query-gpu=name,memory.total --format=csv

Opcional: gravar as execuções no Google Drive, para sobreviverem a uma desconexão.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fedcxr_runs && rm -rf runs && ln -s /content/drive/MyDrive/fedcxr_runs runs

## Dados
Baixa o ChestXRay2017 do Mendeley (CC BY 4.0), confere o SHA-256 e gera `data/prepared/kermany.pt`.

In [ ]:
!python scripts/prepare_data.py --download --raw data/raw --out data/prepared/kermany.pt --workers 2

## Uma execução
FedAvg com ajuste completo, partição Dirichlet α = 0,1, 100 rodadas.

In [ ]:
!python -m fedcxr.train --config configs/base.yaml --set \
    partition.scheme=dirichlet partition.alpha=0.1 fl.method=fedavg model.peft=full fl.lr=0.003 \
    seed=0 train.amp=fp16 train.save_model=false train.out_dir=runs/colab

## Uma grade

`--list` mostra as grades e o número de execuções. A grade `paper` tem 199 execuções e não cabe numa
sessão do Colab; rode grades pequenas (`sens`, `alpha_extra`, `abl_mu`) ou divida a `h2` em várias sessões.
`--per-gpu 2` roda duas execuções ao mesmo tempo na T4.

In [ ]:
!python scripts/run_grid.py --list
!python scripts/run_grid.py --grid sens --gpus 0 --per-gpu 2 --extra train.amp=fp16 train.save_model=false

## Análise
Resumo de uma grade e, com todas as grades em `runs_final/`, as tabelas e figuras do artigo.

In [ ]:
!python scripts/analyze.py --runs runs --grid sens --targets 0.75
# tabelas e figuras do artigo a partir das execuções versionadas no repositório
!python scripts/paper_figures.py --runs runs_final --out saida_artigo